# Project 2 — Goodreads Book Recommender

<small>EDA → collaborative filtering vs a popularity baseline (RMSE + Precision/Recall@N) → an **LLM re-ranking layer** that personalizes the Top-N to a stated preference. Reproducible from `data/Books.csv` and `data/Ratings.csv`. The logic lives in `src/`; this notebook is the narrative that becomes `app.py`.</small>


## Step 0 — Setup


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # so `from src import ...` works from notebooks/
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src import data_loader, cf_model, evaluate, recommend, llm_rerank
pd.set_option('display.max_colwidth', 55)


## Step 1 — Load the data
Reads the two assignment CSVs and repairs the author-name encoding.


In [ ]:
ratings, books = data_loader.load('../data')
print('ratings:', ratings.shape, '| books:', books.shape)
print('users:', ratings.user_id.nunique(), '| rated books:', ratings.book_id.nunique())
books[['book_id','title','authors','original_publication_year','average_rating','ratings_count']].head()


## Step 2 — Exploratory data analysis
Look at users, ratings, and books, and note what each implies for modeling.


In [ ]:
# Rating distribution — skew toward 4–5★ (MNAR: people rate books they expect to like)
rc = ratings['rating'].value_counts().sort_index()
pct_high = 100 * rc.loc[[4.0, 5.0]].sum() / len(ratings)
ax = rc.plot(kind='bar', color='steelblue')
ax.set_title('Rating distribution'); ax.set_xlabel('stars'); ax.set_ylabel('count'); plt.show()
print(ratings['rating'].describe())
print(f'{pct_high:.1f}% of ratings are 4★ or 5★ → strong popularity / mean baseline')


In [ ]:
# Sparsity + activity: ratings per user and per book
n_users, n_books_rated = ratings.user_id.nunique(), ratings.book_id.nunique()
n_cells = n_users * len(books)
sparsity = 1 - len(ratings) / n_cells
per_user = ratings.groupby('user_id').size()
per_book = ratings.groupby('book_id').size()
cold_books = set(books.book_id) - set(ratings.book_id)
print(f'users: {n_users:,} | books in catalog: {len(books):,} | books with ≥1 rating: {n_books_rated:,}')
print(f'ratings/user  -> median {per_user.median():.0f}, max {per_user.max()}')
print(f'ratings/book  -> median {per_book.median():.0f}, max {per_book.max()}')
print(f'matrix sparsity: {sparsity:.2%}  (only {len(ratings)/n_cells:.4%} of cells filled)')
print(f'cold items (never rated in log): {len(cold_books):,} books ({100*len(cold_books)/len(books):.1f}% of catalog)')


In [ ]:
# Popularity long tail: a few books soak up most ratings
ranked = per_book.sort_values(ascending=False).reset_index(drop=True)
ranked.plot(title='Books ranked by #ratings (long tail)', logy=True)
plt.xlabel('book rank'); plt.ylabel('#ratings (log)'); plt.show()
top10_share = ranked.head(10).sum() / ranked.sum()
print(f'Top 10 books account for {100*top10_share:.1f}% of all ratings in the log')


In [ ]:
# Publication year + quality vs popularity; encoding fix sanity check
books['original_publication_year'].dropna().astype(int).hist(bins=30, edgecolor='k')
plt.title('Publication year distribution'); plt.xlabel('year'); plt.show()
books.plot.scatter(x='ratings_count', y='average_rating', alpha=0.2, logx=True,
                   title='average_rating vs ratings_count'); plt.show()
print('language mix:'); print(books['language_code'].value_counts().head())
# Encoding: load() repairs mojibake (latin-1 misread as UTF-8)
raw = pd.read_csv('../data/Books.csv', usecols=['book_id','authors']).set_index('book_id')
print('before load():', raw.loc[2, 'authors'])
print('after  load():', books.loc[2, 'authors'])


### EDA takeaways
- **~68%** of ratings are 4–5★ (MNAR) → popularity/mean is a **strong baseline**; CF must clear it on ranking metrics too.
- Matrix **~98.5% sparse**; median **8** ratings/book → item-based CF struggles on the long tail.
- **735 cold books** (never in the rating log) → content/metadata helps; pure CF cannot recommend them from behavior alone.
- **Popularity long tail** — head books dominate → watch popularity bias and catalog coverage.
- `average_rating` is compressed (~3.5–4.5); weak separation vs `ratings_count`.
- **No shelf tags** in this dataset → content model uses title + authors only (`tags` is empty).
- Author **encoding** needed cleaning (`load(fix_encoding=True)` repairs GrandPré-style mojibake).


## Step 3 — Models: popularity baseline, BaselineOnly, UBCF, IBCF, SVD
Popularity/mean (pure Python) plus `surprise` models. Held-out split (`random_state=6604`) for RMSE and Top-N ranking metrics.


In [ ]:
train, test = evaluate.train_test_split_ratings(ratings, test_size=0.1, seed=6604)

models = {'Popularity': cf_model.PopularityModel().fit(train)}
try:
    cf_model._require_surprise()
    models.update({
        'BaselineOnly': cf_model.CFModel('baseline').fit(train),
        'UBCF': cf_model.CFModel('ubcf').fit(train),
        'IBCF': cf_model.CFModel('ibcf').fit(train),
        'SVD': cf_model.CFModel('svd').fit(train),
    })
except ImportError as e:
    print('scikit-surprise not available — Popularity-only comparison.', e)

comparison = evaluate.compare_cf_models(train, test, models, k=10, threshold=4.0)
comparison


## Step 4 — Interpret the comparison table
**RMSE** = rating accuracy on held-out stars. **Precision/Recall/NDCG@10** = Top-N quality (relevant = test rating ≥ 4). They often disagree — tune and report both.


In [ ]:
# Pick the best CF model by NDCG@10 (change metric if your story differs)
best_name = comparison['NDCG@10'].idxmax()
best = models[best_name]
print(f'Best on NDCG@10: {best_name}')
comparison.sort_values('NDCG@10', ascending=False)


**Interpretation (write this up).** If the popularity baseline is hard to beat, that's the expected outcome on a sparse, popularity-skewed dataset — explain *why* (sparsity, cold items, head dominance) and what it implies for model selection. This honest analysis is worth points.


## Step 5 (optional extra) — Hybrid: TF-IDF vs semantic embeddings
Content vectors live **in memory** (no ChromaDB). `backend="embeddings"` needs `pip install sentence-transformers`.

In [ ]:
from src.content_model import ContentModel
from src.hybrid import HybridRecommender

content_tfidf = ContentModel(backend='tfidf').fit(books)
hybrid_tfidf = HybridRecommender(cf_model=best, content_model=content_tfidf, alpha=0.6)

uid_h = ratings.user_id.value_counts().index[0]
ur = ratings[ratings.user_id == uid_h]
cands = [b for b in books.book_id if b not in set(ur.book_id)]
hybrid_scores = hybrid_tfidf.score(uid_h, ur, cands).sort_values(ascending=False).head(5)
print('Hybrid (TF-IDF +', best_name, '):')
print(hybrid_scores)

try:
    content_emb = ContentModel(backend='embeddings').fit(books)
    hybrid_emb = HybridRecommender(cf_model=best, content_model=content_emb, alpha=0.6)
    emb_scores = hybrid_emb.score(uid_h, ur, cands).sort_values(ascending=False).head(5)
    print('\nHybrid (sentence-transformers +', best_name, '):')
    print(emb_scores)
except ImportError:
    print('\nSkipping embeddings backend — install sentence-transformers to compare.')

## Step 6 — Top-N from the best CF model, then the LLM layer
Pick the best model, generate a user's Top-N, then re-rank to a stated preference.


In [ ]:
# `best` and `best_name` come from Step 4

class A:  # tiny adapter so recommend_top_n can use the raw model
    content = None
    def __init__(s, m): s.m = m
    def score(s, u, ur, ids): return s.m.predict_for_user(u, ids)

uid = ratings.user_id.value_counts().index[0]   # an active user
cf_recs = recommend.recommend_top_n(uid, A(best), ratings, books, top_n=10, min_ratings=50, explain=False)
cf_recs[['book_id','title','authors','score']]


In [ ]:
# LLM re-ranking: candidates -> Gemini (if GEMINI_API_KEY set) else heuristic fallback.
# Set your key in the environment first (never in code):  export GEMINI_API_KEY=...
cands = llm_rerank.candidates_from_recs(cf_recs, books)
picks, used_llm = llm_rerank.rerank(cands, preference='something dark and mysterious', top_k=5)
print('source:', 'Gemini LLM' if used_llm else 'heuristic fallback (no key)')
for i, p in enumerate(picks, 1):
    print(f'{i}. {p.title} — {p.authors}')
    print(f'   why: {p.explanation}')


## Step 7 — From notebook to Streamlit
This exact flow (pick user → CF Top-N → LLM re-rank) is wired in `../app.py`:

```bash
cd ..
export GEMINI_API_KEY=your_key_here   # optional; without it the app uses the fallback
streamlit run app.py
```

See `PROJECT_FRAMEWORK.md` for the rubric map and `docs/LLM_RERANK.md` for the grounded prompt strategy.

**Vectors:** content similarity stays in memory (`content_model.py`) — no ChromaDB at this catalog size.
